# 05: AM defense (PLAN.md §6, §13.7)
Build AM on BASE, choose its recipe **on dev only**, freeze it, then run the normal A0-A5 battery on test.

1. **locate**: protected layers L* from three per-layer rankings (refusal-direction strength, SafeTune safety
   neurons, CircuitKIT EAP-IG; CPU fallback), merged by mean rank. Saves the method overlap for the write-up.
2. **harden** (6 candidates = k in {4,6,8} x spread on/off): SafeTune SafeGrad, LoRA only on L*, 300 refusal +
   300 benign pairs. Spread-refusal ablates BASE's top refusal direction on half the refusal batches.
   Every candidate carries the compression shield (L* at 8-bit under A4).
3. **dev evaluation**: A0, A2, A3, A4q3 on dev splits, judged with WildGuard.
4. **select stage 1**: best mean retention with benign_dev over-refusal <= BASE + 3 pts and mmlu_dev >= BASE - 2 pts.
5. **ReSta** re-anchor on the winner, kept only if eligible and strictly better on dev (stage 2).
6. **controls** with the chosen recipe and the same budget: AM_rand0-2 (random layers outside L*), AM_all.
7. **freeze**: AM + controls pushed to the PRIVATE store `<you>/aftermarket-am`, revision pinned in
   `configs/am_frozen.yaml`. Only then the test battery (`--final`).

Safe for **Save & Run All (Commit)**: every stage skips finished work and pulls from the private Hub first, so a
re-commit resumes. If no candidate is eligible, freezing stops and nothing runs on test (report it honestly).
Defense weights live in `/kaggle/temp` (not in the notebook output). Attack adapters are never pushed.

Requires: `HF_TOKEN` (write), GPU T4 x2, ~8-10 h in total (resumable).

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
DDIR = os.environ["AFTERMARKET_DEFENSE_DIR"]
def read_json(path):
    try:
        return json.load(open(path))
    except Exception:
        return None
print("defense dir:", DDIR)

## 1. Locate L*

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.locate --config configs/defense.yaml

## 2. Harden: six candidates, one per GPU at a time

In [ ]:
# pull once, then two trainers in parallel (one per T4); each skips candidates that already exist
!PYTHONPATH=src python -c "import yaml; from aftermarket.defense import store; store.pull_all(yaml.safe_load(open('configs/defense.yaml')))"
!(CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m aftermarket.defense.harden --config configs/defense.yaml --stage candidates --shard 0/2 --no-pull > /kaggle/temp/harden0.log 2>&1 & CUDA_VISIBLE_DEVICES=1 PYTHONPATH=src python -m aftermarket.defense.harden --config configs/defense.yaml --stage candidates --shard 1/2 --no-pull > /kaggle/temp/harden1.log 2>&1 & wait); tail -n 25 /kaggle/temp/harden0.log /kaggle/temp/harden1.log

## 3. Dev evaluation (A0, then A3 + A4q3, then A2) and WildGuard judging

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_a0.yaml

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_attacks.yaml --attacks A3,A4q3

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_attacks.yaml --attacks A2

In [ ]:
!PYTHONPATH=src python -m aftermarket.judge --config configs/judge_defense.yaml

## 4. Selection, stage 1 (k x spread)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.select --config configs/defense.yaml --stage 1

## 5. ReSta re-anchor on the winner, evaluated the same way (stage 2)

In [ ]:
sel = read_json(f"{DDIR}/selection.json") or {}
CHOSEN = (sel.get("stage1") or {}).get("chosen")
print("stage-1 winner:", CHOSEN)
if CHOSEN:
    !PYTHONPATH=src python -m aftermarket.defense.harden --config configs/defense.yaml --stage resta
    !PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_a0.yaml --runs {CHOSEN}_r
    !PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_attacks.yaml --runs {CHOSEN}_r --attacks A3,A4q3
    !PYTHONPATH=src python -m aftermarket.generate --config configs/gen_defense_dev_attacks.yaml --runs {CHOSEN}_r --attacks A2
    !PYTHONPATH=src python -m aftermarket.judge --config configs/judge_defense.yaml
    !PYTHONPATH=src python -m aftermarket.defense.select --config configs/defense.yaml --stage 2
else:
    print("no eligible candidate: stopping before ReSta / controls / freeze")

## 6. Controls with the chosen recipe (same budget), then 7. freeze

In [ ]:
sel = read_json(f"{DDIR}/selection.json") or {}
if sel.get("recipe") and "stage2" in sel:
    !PYTHONPATH=src python -c "import yaml; from aftermarket.defense import store; store.pull_all(yaml.safe_load(open('configs/defense.yaml')))"
    !(CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m aftermarket.defense.harden --config configs/defense.yaml --stage controls --shard 0/2 --no-pull > /kaggle/temp/controls0.log 2>&1 & CUDA_VISIBLE_DEVICES=1 PYTHONPATH=src python -m aftermarket.defense.harden --config configs/defense.yaml --stage controls --shard 1/2 --no-pull > /kaggle/temp/controls1.log 2>&1 & wait); tail -n 15 /kaggle/temp/controls0.log /kaggle/temp/controls1.log
    if not os.path.exists("configs/am_frozen.yaml"):
        !PYTHONPATH=src python -m aftermarket.defense.select --config configs/defense.yaml --stage install
    if not os.path.exists("configs/am_frozen.yaml"):
        !PYTHONPATH=src python -m aftermarket.defense.select --config configs/defense.yaml --stage freeze
else:
    print("selection incomplete: no controls, no freeze")

## 8. Frozen AM + controls through the A0-A5 battery on test (`--final`), cheap attacks first

In [ ]:
if os.path.exists("configs/am_frozen.yaml"):
    for stage in ["A0,A3,A5raw", "A4q4,A4q3,A4q2", "A5bon", "A1", "A2"]:
        !PYTHONPATH=src python -m aftermarket.generate --config configs/gen_am_test.yaml --final --attacks {stage}
    !PYTHONPATH=src python -m aftermarket.judge --config configs/judge.yaml
    !PYTHONPATH=src python -m aftermarket.generate --config configs/gen_am_test.yaml --push-pending
else:
    print("AM is not frozen: no test run")

In [ ]:
# Final summary (always runs; stats only, no completion text)
try:
    loc = read_json(f"{DDIR}/locate/locate.json")
    if loc:
        print("locate methods:", loc["methods_used"], "| failures:", loc["failures"] or "none")
        for k, L in loc["L_star"].items():
            print(f"  L*_{k} = {L} | overlap {loc['overlap'][k]['jaccard']} | in all: {loc['overlap'][k]['in_all']}")
    else:
        print("locate: not done")
    from aftermarket.defense.store import local_specs
    specs = local_specs()
    for mid, s in specs.items():
        m = s["defense"]
        print(f"  {mid:12s} {m.get('kind')} layers={m.get('layers') if len(m.get('layers') or []) <= 8 else 'all'} "
              f"spread={m.get('spread')} params={m.get('trainable_params')} train_s={m.get('train_seconds')} "
              f"loss {m.get('loss_first')} -> {m.get('loss_last')} shield={bool(m.get('quant_shield'))}")
    sel = read_json(f"{DDIR}/selection.json")
    if sel:
        for t in sel["stage1"]["table"] + (sel.get("stage2") or {}).get("table", []):
            print(f"  {t['id']:12s} retention {t['mean_retention']} | A0 ASR {t['asr_A0']} | over-refusal "
                  f"{t['over_refusal']} | mmlu {t['mmlu']} | {'eligible' if t['eligible'] else 'x ' + ','.join(t['why_not'])}")
        print("recipe:", sel.get("recipe"))
    if os.path.exists("configs/am_frozen.yaml"):
        import yaml
        fz = yaml.safe_load(open("configs/am_frozen.yaml"))
        print("FROZEN:", fz["recipe"], "| L* =", fz["L_star"], "|", fz["store"])
        print("commit configs/am_frozen.yaml to git (content below):")
        print(open("configs/am_frozen.yaml").read())
        from pathlib import Path
        from aftermarket.generate import make_jobs, pending_jobs
        c = yaml.safe_load(open("configs/gen_am_test.yaml"))
        jobs = make_jobs(c)
        print(f"AM test battery: {len(jobs) - len(pending_jobs(jobs, Path(c['out_dir'])))}/{len(jobs)} jobs done")
    else:
        print("AM not frozen yet")
except Exception as e:
    print(f"final summary failed: {type(e).__name__}: {e}")